In [ ]:
import os
import numpy as np
from scipy.io import loadmat
from scipy import stats, signal
from scipy.signal import resample_poly, convolve
import matplotlib.pyplot as plt
from fractions import Fraction
import pandas as pd

Functions for later

In [ ]:
def zscore_data(data, n_runs):
    """
    Z-score normalization across time points for each run and subject.
    """
    if n_runs > 1:
        return stats.zscore(data, axis=2, nan_policy='omit')
    else:
        return stats.zscore(data, axis=1, nan_policy='omit')

def interpolate_nans(data, n_runs):
    """
    Interpolates over NaN values in a time series after Z-scoring.
    """

    if n_runs > 1:
        n_subj, n_runs, n_TRs = data.shape
        interp_data = np.copy(data)
    
        for subj in range(n_subj):
            for run in range(n_runs):
                y = data[subj, run, :]
                nans = np.isnan(y)
                x = np.arange(len(y))
    
                if np.any(nans):
                    # Perform linear interpolation over NaNs
                    y[nans] = np.interp(x[nans], x[~nans], y[~nans])
    
                interp_data[subj, run, :] = y

    else:
        n_subj, n_TRs = data.shape
        interp_data = np.copy(data)
    
        for subj in range(n_subj):
                y = data[subj, :]
                nans = np.isnan(y)
                x = np.arange(len(y))
    
                if np.any(nans):
                    # Perform linear interpolation over NaNs
                    y[nans] = np.interp(x[nans], x[~nans], y[~nans])
    
                interp_data[subj, :] = y

    return interp_data

def resample_hrf(hrf_kernel, TR, max_denominator=100):
    """
    Resamples the HRF kernel to match the TR length.

    Parameters:
        hrf_kernel (numpy array): The original HRF kernel.
        TR (float): The repetition time (TR) in seconds.
        max_denominator (int): Maximum denominator for rational approximation.

    Returns:
        numpy array: The resampled HRF kernel.
    """
    if TR.is_integer():
        hrf_resampled = resample_poly(hrf_kernel, 1, int(TR))
    else:
        frac = Fraction(TR).limit_denominator(max_denominator)
        num, denom = frac.numerator, frac.denominator

        print(f"Resampling HRF with ratio: {num}/{denom}")

        hrf_resampled = resample_poly(hrf_kernel, num, denom)

    return hrf_resampled / np.sum(hrf_resampled)

def convolve_hrf(time_series, hrf_kernel, TR, n_runs, do_convolve=True):
    """
    Convolves a time-series with the HRF kernel separately for each subject and run.

    Parameters:
        time_series (numpy array): Input time-series (n_subj, n_runs, n_TRs) or (n_subj, n_TRs).
        hrf_kernel (numpy array): Hemodynamic response function (HRF).
        TR (float): Repetition time (TR).
        n_runs (int): Number of runs.
        do_convolve (bool): Whether to apply HRF convolution. Defaults to True.

    Returns:
        numpy array: Convolved and trimmed time-series (same shape as input).
    """
    if not do_convolve:
        return time_series

    hrf_resampled = resample_hrf(hrf_kernel, TR)

    if n_runs > 1:
        if time_series.ndim != 3:
            raise ValueError(f"Expected shape (n_subj, n_runs, n_TRs) for multiple runs, got {time_series.shape}")
        n_subj, n_runs, n_TRs = time_series.shape
    else:
        if time_series.ndim == 2:
            time_series = time_series[:, np.newaxis, :]
        elif time_series.ndim != 3:
            raise ValueError(f"Expected shape (n_subj, n_TRs) or (n_subj, n_runs, n_TRs), got {time_series.shape}")
        n_subj, _, n_TRs = time_series.shape

    convolved_signal = np.zeros_like(time_series)

    for subj in range(n_subj):
        for run in range(n_runs):
            convolved = convolve(time_series[subj, run, :], hrf_resampled, mode='full')[:n_TRs]
            convolved_signal[subj, run, :] = convolved

    if n_runs == 1:
        convolved_signal = convolved_signal[:, 0, :]

    return convolved_signal

def smooth_data(data, n_runs, TR, smooth_duration=30):
    window_size = max(1, int(smooth_duration / TR))
    print(f" Window size is {window_size}")

    gaussian_window = signal.windows.gaussian(window_size, std=window_size / 4)
    gaussian_window /= np.sum(gaussian_window)

    smoothed_data = np.zeros_like(data)

    if n_runs > 1:
        n_subj, n_runs, n_TRs = data.shape
        for subj in range(n_subj):
            for run in range(n_runs):
                smoothed = np.convolve(data[subj, run, :], gaussian_window, mode='same')
                smoothed_data[subj, run, :] = smoothed
    else:
        n_subj, n_TRs = data.shape
        for subj in range(n_subj):
            smoothed = np.convolve(data[subj, :], gaussian_window, mode='same')
            smoothed_data[subj, :] = smoothed

    return smoothed_data

def trim_edges(data, trim_size, n_runs):
    if n_runs > 1:
        n_subj, n_runs, n_TRs = data.shape
        trimmed_data = data[:, :, trim_size:-trim_size]
    else:
        n_subj, n_TRs = data.shape
        trimmed_data = data[:, trim_size:-trim_size]
    return trimmed_data

def process_spectral_slopes_per_run(spectral_slopes, hrf_kernel, TR, smooth_duration=30, z_score=True, n_runs=2, do_convolve=True, do_trim = True):
    """
    Full processing pipeline per run: Z-score, interpolate, convolve, smooth, and trim.
    """
    print("Starting Per-Run Spectral Slope Processing Pipeline...")

    if z_score:
        data = zscore_data(spectral_slopes, n_runs)
    else:
        data = spectral_slopes

    interp_data = interpolate_nans(data, n_runs)
    convolved_data = convolve_hrf(interp_data, hrf_kernel, TR=TR, n_runs=n_runs, do_convolve=do_convolve)
    smoothed_data = smooth_data(convolved_data, n_runs=n_runs, TR=TR, smooth_duration=smooth_duration)

    if do_trim:
        trim_size = max(1, int(smooth_duration / (2 * TR)))
        trimmed_data = trim_edges(smoothed_data, trim_size=trim_size, n_runs=n_runs)
    else:
        trimmed_data = smoothed_data

    if n_runs > 1:
        n_subj, n_runs, n_TRs_trimmed = trimmed_data.shape
        n_total_TRs = n_runs * n_TRs_trimmed
        combined_data = trimmed_data.reshape(n_subj, n_total_TRs)
    else:
        combined_data = trimmed_data

    print("Per-Run Processing Complete.")
    return combined_data


def plot_smoothed_data(data_array, subject_idx):
    """
    Plot smoothed time series data for a single subject.

    Parameters:
    -----------
    data_array : np.ndarray
        Array of shape (n_subjects, n_TRs), containing the smoothed signal.
    subject_idx : int
        Index of the subject to plot.
    """
    if subject_idx < 0 or subject_idx >= data_array.shape[0]:
        raise IndexError(f"Subject index {subject_idx} out of range for data with shape {data_array.shape}")

    time_series = data_array[subject_idx]

    plt.figure(figsize=(12, 4))
    plt.plot(time_series, linewidth=1.5)
    plt.title(f"Smoothed Time Series - Subject {subject_idx}", fontsize=14)
    plt.xlabel("TRs", fontsize=12)
    plt.ylabel("Smoothed Signal", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


Preprocess Pupil Rest

In [ ]:
# ---- Parameters ---- #
# Path to the directory containing subject folders
data_dir = "../1_data/2_Pupil_Rest_Arousal/"
# List of subject IDs
all_sub = ["1372", "1387", "1393", "1416", "1560", "1582", "1604", "1674", "1700",
           "1766", "1792", "1811", "1834", "1841", "1859", "1913", "1940", "1978", 
           "2079", "2100", "2106", "2196", "2213", "2216", "2220", "2221", "2226"]

# ---- Initialize List to Store Data ---- #
pupil_data_list = []

# ---- Read Data ---- #
for subj_id in all_sub:
    subj_folder = f"sub-pa{subj_id}"
    subj_data = []  # To store the two runs for this subject

    for run_num in [1, 2]:
        # Build file path for each run
        file_name = f"sub-pa{subj_id}_task-rest_run-0{run_num}_et.tsv"
        file_path = os.path.join(data_dir, subj_folder, file_name)

        # Check if file exists
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            continue

        # Read data (assuming one column of pupil data, one value per row)
        try:
            run_data = np.loadtxt(file_path)
            subj_data.append(run_data)
            print(f"Loaded: {file_path} | Shape: {run_data.shape}")
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    # Stack the two runs into (n_runs, n_TRs) if both runs are loaded
    if len(subj_data) == 2:
        subj_array = np.stack(subj_data, axis=0)  # Shape: (2, n_TRs)
        pupil_data_list.append(subj_array)
    else:
        print(f"Subject {subj_id} missing one or more runs, skipping.")

# ---- Convert to NumPy Array ---- #
# Final shape will be (n_subj, n_runs, n_TRs)
pupil_data_array = np.array(pupil_data_list)
print(f"Final Pupil Data Shape: {pupil_data_array.shape}")  # Expecting (n_subj, 2, n_TRs)

# ---- Save as .npy File ---- #
output_file = "../1_data/2_Pupil_Rest_Arousal/Pupil_Rest_Arousal.npy"
np.save(output_file, pupil_data_array)
print(f"Pupil data saved to {output_file}")


Extract and preprocess Pupil Story

In [ ]:
# Set directory containing CSV files
pupil_dir = "../1_data/3_Pupil_Story_Arousal/"  # Update with actual path

# List all CSV files (each file corresponds to one subject)
pupil_files = sorted([f for f in os.listdir(pupil_dir) if f.endswith(".csv")])

# Define run segment sizes
run_sizes = [524, 449, 329]  # (run1, run2, run3)
total_samples = sum(run_sizes)  # 1302

# Ensure all files contain the expected number of rows (excluding header)
subject_data_run1 = []
subject_data_run2 = []
subject_data_run3 = []

for file in pupil_files:
    # Load data (assumes first row is header)
    file_path = os.path.join(pupil_dir, file)
    df = pd.read_csv(file_path, header=0)  # Load CSV, skipping header row
    
    # Extract pupil values (assume first column contains pupil data)
    pupil_values = df['pupilSize'].values

    if len(pupil_values) == 1303:
        pupil_values = pupil_values[:-1]
    
    
    # Ensure correct length
    if len(pupil_values) != total_samples:
        print(f"⚠️ Warning: {file} has {len(pupil_values)} values instead of {total_samples}. Skipping.")
        continue
    
    # Split into runs
    run1, run2, run3 = np.split(pupil_values, [run_sizes[0], run_sizes[0] + run_sizes[1]])
    
    # Store raw runs
    subject_data_run1.append(run1)
    subject_data_run2.append(run2)
    subject_data_run3.append(run3)

# Convert lists to NumPy arrays (n_subjects, n_TRs_per_run)
subject_data_run1 = np.array(subject_data_run1)#[:,30:]
subject_data_run2 = np.array(subject_data_run2)#[:,30:]
subject_data_run3 = np.array(subject_data_run3)#[:,30:]


# **Apply z-score using zscore_data function**
zscored_run1 = zscore_data(subject_data_run1, n_runs=1)  # (n_subjects, n_TRs_per_run)
zscored_run2 = zscore_data(subject_data_run2, n_runs=1)  # (n_subjects, n_TRs_per_run)
zscored_run3 = zscore_data(subject_data_run3, n_runs=1)  # (n_subjects, n_TRs_per_run)

# Compute the average pupil response across subjects for each run
avg_run1 = np.mean(zscored_run1, axis=0).reshape(1,-1)
avg_run2 = np.mean(zscored_run2, axis=0).reshape(1,-1)
avg_run3 = np.mean(zscored_run3, axis=0).reshape(1,-1)

print(avg_run1.shape)
print(avg_run2.shape)
print(avg_run3.shape)


Now lets preprocess and convolve our arousal data! 

In [ ]:
# ---- Step 1: Load Data ---- #
spectral_slope_array_05Hz_80Hz = np.load("../1_data/1_EEG_Arousal/SpectralSlopes_05Hz_80Hz.npy")
pupil_array = np.load("../1_data/2_Pupil_Rest_Arousal/Pupil_Rest_Arousal.npy")


# Define HRF kernel (from MATLAB script)
hrf_kernel = np.array([0, 0.000354107958396228, 0.0220818694830938, 0.116001537001027,
                        0.221299059999514, 0.242353095826523, 0.186831750619196,
                        0.113041009515928, 0.0572809597709863, 0.0253492394574127,
                        0.0100814114758446, 0.00367740475539297, 0.00124901102357508,
                        0.000399543113110135, 0])

# Ensure HRF is normalized
hrf_kernel /= np.sum(hrf_kernel)  # Normalize the HRF kernel so that it sums to 1

# ---- Run Processing Pipeline ---- #
processed_eeg_data_05Hz_80Hz = process_spectral_slopes_per_run(spectral_slope_array_05Hz_80Hz, hrf_kernel, TR=2.1)
processed_pupil_data = process_spectral_slopes_per_run(pupil_array, hrf_kernel, TR=1)

processed_paranoia_run1 = process_spectral_slopes_per_run(avg_run1, hrf_kernel, TR=1, n_runs=1, z_score=False)
processed_paranoia_run2 = process_spectral_slopes_per_run(avg_run2, hrf_kernel, TR=1, n_runs=1, z_score=False)
processed_paranoia_run3 = process_spectral_slopes_per_run(avg_run3, hrf_kernel, TR=1, n_runs=1, z_score=False)

# Remove extra dimension (convert from (1, n_TRs) → (n_TRs,))
processed_paranoia_run1 = processed_paranoia_run1.squeeze()  # Shape: (524,)
processed_paranoia_run2 = processed_paranoia_run2.squeeze()  # Shape: (449,)
processed_paranoia_run3 = processed_paranoia_run3.squeeze()  # Shape: (329,)

# Combine runs in order (Run1 → Run2 → Run3)
final_processed_paranoia = np.concatenate([processed_paranoia_run1, processed_paranoia_run2, processed_paranoia_run3])  # Shape: (1302,)

# Reshape to (1, 1302) to maintain (n_subj, n_TRs) structure
final_processed_paranoia = final_processed_paranoia.reshape(1, -1)  # Shape: (1, 1302)

# Save the processed data
np.save("../4_results/1_arousal/1_EEG_Rest/Convolved_EEG_Rest_Arousal.npy", processed_eeg_data_05Hz_80Hz)
np.save("../4_results/1_arousal/2_Pupil_Rest/Convolved_Pupil_Rest_Arousal.npy", processed_pupil_data)
np.save("../4_results/1_arousal/3_Pupil_Story/Convolved_Pupil_Story_Arousal.npy", final_processed_paranoia)
print(f"Processing EEG Rest complete with outputted array shape: {processed_eeg_data_05Hz_80Hz.shape}!")
print(f"Processing Pupil Rest complete with outputted array shape: {processed_pupil_data.shape}!")
print(f"Processing Pupil Story complete with outputted array shape: {final_processed_paranoia.shape}!")

Calculating and saving average self report arousal score for Filmfestival and Sherlock events

In [ ]:
# Load the CSV, skipping the first row (subject IDs)
df = pd.read_csv("../1_data/4_FilmFestival_Arousal/FilmFest_arousal_discrete_allSubj.csv", header=None, skiprows=1)

# Compute row-wise means (i.e., average score per event)
event_means = df.mean(axis=1, skipna=True)  # skipna handles any missing subject scores


np.save("../4_results/1_arousal/4_FilmFestival/Filmfest_Event_Arousal.npy", event_means)

In [ ]:
# Load the CSV, skipping the first row (subject IDs)
df = pd.read_csv("../1_data/5_Sherlock_Arousal/Sherlock_arousal_discrete_allSubj.csv", header=None, skiprows=1)

# Compute row-wise means (i.e., average score per event)
event_means = df.mean(axis=1, skipna=True)  # skipna handles any missing subject scores

np.save("../4_results/1_arousal/5_Sherlock/Sherlock_Event_Arousal.npy", event_means)
